# Import Libraries

In [1]:
import os
import time
import random
import xml.etree.ElementTree as ET
from collections import OrderedDict

import numpy as np
from PIL import Image
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import torch.nn as nn

# Kaggle GPU Safety Check

In [2]:
def kaggle_gpu_guard():
    if not torch.cuda.is_available():
        print("CUDA not available. Running on CPU.")
        return torch.device("cpu"), False

    gpu_name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)

    print("GPU:", gpu_name)
    print("Compute capability:", cc)
    print("Torch:", torch.__version__)
    print("CUDA:", torch.version.cuda)

    if "P100" in gpu_name and cc == (6, 0):
        print("Detected Kaggle P100 (sm_60).")
        print("This runtime may fail with this Torch/CUDA build.")
        print("Falling back to CPU.")
        return torch.device("cpu"), False

    return torch.device("cuda"), True


DEVICE, USE_AMP = kaggle_gpu_guard()

GPU: Tesla T4
Compute capability: (7, 5)
Torch: 2.10.0+cu128
CUDA: 12.8


# Experiment Configuration

In [3]:
class CFG:
    SEED = 42
    VOC_ROOT = "/kaggle/input/datasets/icaslab/object-detection-assignment-2026/PASCAL_Object_Detection/VOC2012_train_val/VOC2012_train_val"

    # True = baseline with pretrained MobileNet backbone
    # False = student custom backbone
    USE_PRETRAINED_BASELINE = True

    # Split strategy
    TEST_SPLIT = "test" 
    VAL_RATIO = 0.1        # split original train -> 90% train, 10% val

    EPOCHS = 1
    BATCH_SIZE = 16
    NUM_WORKERS = 2        # safer in notebooks; set 0 if debugging
    LR = 1e-4
    WEIGHT_DECAY = 1e-4

    MIN_SIZE = 384
    MAX_SIZE = 384
    FIXED_OUT_CHANNELS = 128

    DEVICE = DEVICE
    USE_AMP = USE_AMP

    OUTPUT_DIR = "/kaggle/working"
    SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits")
    BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_voc_detector.pth")
    FINAL_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, "final_model_weights_only.pth")
    SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

# VOC Class Names and Reproducibility

In [4]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

CLASS_TO_IDX = {cls_name: i + 1 for i, cls_name in enumerate(VOC_CLASSES)}
IDX_TO_CLASS = {i + 1: cls_name for i, cls_name in enumerate(VOC_CLASSES)}
IDX_TO_CLASS[0] = "__background__"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def parse_voc_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    boxes = []
    labels = []

    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip().lower()
        if cls_name not in CLASS_TO_IDX:
            continue

        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        if xmax <= xmin or ymax <= ymin:
            continue

        boxes.append([xmin, ymin, xmax, ymax])
        labels.append(CLASS_TO_IDX[cls_name])

    if len(boxes) == 0:
        boxes = torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.zeros((0,), dtype=torch.int64)
    else:
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

    return boxes, labels

set_seed(CFG.SEED)

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", CFG.DEVICE)
print("VOC root exists:", os.path.exists(CFG.VOC_ROOT))

Torch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Device: cuda
VOC root exists: True


# Data Augmentation for Detection

In [5]:
class DetectionTransform:
    def __init__(self, train=True):
        self.train = train

    def __call__(self, image, target=None):
        image = F.to_tensor(image)

        if self.train and target is not None and random.random() < 0.5:
            image = torch.flip(image, dims=[2])
            width = image.shape[2]
            boxes = target["boxes"]

            if boxes.numel() > 0:
                x1 = boxes[:, 0].clone()
                x2 = boxes[:, 2].clone()
                boxes[:, 0] = width - x2
                boxes[:, 2] = width - x1
                target["boxes"] = boxes

        return image, target

# Pascal VOC Dataset Loader

In [6]:
original_train_file = os.path.join(CFG.VOC_ROOT, "ImageSets", "Main", "train.txt")

with open(original_train_file, "r") as f:
    original_train_ids = [line.strip() for line in f.readlines() if line.strip()]

random.seed(CFG.SEED)
random.shuffle(original_train_ids)

n_val = int(len(original_train_ids) * CFG.VAL_RATIO)

val_ids = sorted(original_train_ids[:n_val])
train_ids_new = sorted(original_train_ids[n_val:])

print("Train samples:", len(train_ids_new))
print("Val samples  :", len(val_ids))

Train samples: 5146
Val samples  : 571


In [7]:
class VOCDataset(Dataset):
    def __init__(self, voc_root, split="train", transforms=None):
        self.voc_root = voc_root
        self.transforms = transforms

        self.annotations_dir = os.path.join(voc_root, "Annotations")
        self.images_dir = os.path.join(voc_root, "JPEGImages")
        self.splits_dir = os.path.join(voc_root, "ImageSets", "Main")

        if isinstance(split, list):
            self.image_ids = split
        elif os.path.isfile(split):
            with open(split, "r") as f:
                self.image_ids = [line.strip() for line in f.readlines() if line.strip()]
        else:
            split_file = os.path.join(self.splits_dir, f"{split}.txt")
            if not os.path.exists(split_file):
                raise FileNotFoundError(f"Split file not found: {split_file}")
            with open(split_file, "r") as f:
                self.image_ids = [line.strip() for line in f.readlines() if line.strip()]

        self.class_names = ["__background__"] + VOC_CLASSES

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        img_path = os.path.join(self.images_dir, f"{image_id}.jpg")
        ann_path = os.path.join(self.annotations_dir, f"{image_id}.xml")

        image = Image.open(img_path).convert("RGB")
        boxes, labels = parse_voc_xml(ann_path)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([int(image_id)], dtype=torch.int64)
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target

In [8]:
class VOCImageOnlyDataset(Dataset):
    def __init__(self, voc_root, split="val", transforms=None):
        self.voc_root = voc_root
        self.transforms = transforms

        self.images_dir = os.path.join(voc_root, "JPEGImages")
        self.splits_dir = os.path.join(voc_root, "ImageSets", "Main")

        split_file = os.path.join(self.splits_dir, f"{split}.txt")
        if not os.path.exists(split_file):
            raise FileNotFoundError(f"Split file not found: {split_file}")

        with open(split_file, "r") as f:
            self.image_ids = [line.strip() for line in f.readlines() if line.strip()]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = os.path.join(self.images_dir, f"{image_id}.jpg")

        image = Image.open(img_path).convert("RGB")

        if self.transforms is not None:
            image, _ = self.transforms(image, None)

        return image, image_id

# Batch Collation for Variable-Sized Detection Targets

In [9]:
def collate_fn(batch):
    return tuple(zip(*batch))

def collate_fn_test(batch):
    images, image_ids = zip(*batch)
    return list(images), list(image_ids)

# Pretrained version

In [10]:
# ===============================
# Pretrained StudentNet
# ===============================

class StudentNet_Pretrained(nn.Module):
    def __init__(self):
        super().__init__()

        base_model = mobilenet_v3_small(
            weights=MobileNet_V3_Small_Weights.DEFAULT
        )

        self.features = base_model.features
        self.out_channels = 576

    def forward(self, x):
        return self.features(x)

# Student Backbone Template

In [11]:
# ===============================
# Custom StudentNet (Editable)
# ===============================

class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Students modify this
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.out_channels = 128  # MUST match last layer

    def forward(self, x):
        return self.features(x)

# Fixed Wrapper to Connect Backbone to Faster R-CNN 

In [12]:
class DetectorBackbone(nn.Module):
    def __init__(self, student_backbone, fixed_out_channels=128):
        super().__init__()
        self.features = student_backbone
        self.adapter = nn.Conv2d(
            student_backbone.out_channels,
            fixed_out_channels,
            kernel_size=1
        )
        self.out_channels = fixed_out_channels

    def forward(self, x):
        x = self.features(x)
        x = self.adapter(x)
        return OrderedDict([("0", x)])

# Parameter Breakdown Utility

In [13]:
def count_param_breakdown(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    student_params = sum(p.numel() for p in model.backbone.features.parameters())
    adapter_params = sum(p.numel() for p in model.backbone.adapter.parameters())
    backbone_total = sum(p.numel() for p in model.backbone.parameters())
    head_params = total_params - backbone_total

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "student_backbone_params": student_params,
        "adapter_params": adapter_params,
        "backbone_total_params": backbone_total,
        "head_params": head_params,
    }

# Faster R-CNN Model Builder

In [14]:
def build_model(num_classes, min_size=512, max_size=512, fixed_out_channels=128):
    
    # Select backbone
    if CFG.USE_PRETRAINED_BASELINE:
        student_backbone = StudentNet_Pretrained()
        print("Using PRETRAINED backbone (baseline)")
    else:
        student_backbone = StudentNet()
        print("Using CUSTOM student backbone")

    backbone = DetectorBackbone(
        student_backbone=student_backbone,
        fixed_out_channels=fixed_out_channels,
    )

    anchor_generator = AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),)
    )

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=["0"],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        min_size=min_size,
        max_size=max_size
    )

    return model

# Build the Datasets and DataLoaders

In [15]:
train_dataset = VOCDataset(
    voc_root=CFG.VOC_ROOT,
    split=train_ids_new, 
    transforms=DetectionTransform(train=True),
)

val_dataset = VOCDataset(
    voc_root=CFG.VOC_ROOT,
    split=val_ids,      
    transforms=DetectionTransform(train=False),
)

test_dataset = VOCImageOnlyDataset(
    voc_root=CFG.VOC_ROOT,
    split=CFG.TEST_SPLIT,  # still "val"
    transforms=DetectionTransform(train=False),
)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(CFG.DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(CFG.DEVICE.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn_test,
    pin_memory=(CFG.DEVICE.type == "cuda"),
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Hidden test size:", len(test_dataset))
print("Classes:", train_dataset.class_names)

Train size: 5146
Val size: 571
Hidden test size: 5823
Classes: ['__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']


# Build the Detector and Inspect Parameter Counts

In [16]:
model = build_model(
    num_classes=len(train_dataset.class_names),
    min_size=CFG.MIN_SIZE,
    max_size=CFG.MAX_SIZE,
    fixed_out_channels=CFG.FIXED_OUT_CHANNELS,
)
model.to(CFG.DEVICE)

stats = count_param_breakdown(model)

print("Total params            :", f"{stats['total_params']:,}")
print("Trainable params        :", f"{stats['trainable_params']:,}")
print("Student backbone params :", f"{stats['student_backbone_params']:,}")
print("Adapter params          :", f"{stats['adapter_params']:,}")
print("Backbone total params   :", f"{stats['backbone_total_params']:,}")
print("Head params             :", f"{stats['head_params']:,}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 108MB/s]


Using PRETRAINED backbone (baseline)
Total params            : 8,738,900
Trainable params        : 8,738,900
Student backbone params : 927,008
Adapter params          : 73,856
Backbone total params   : 1,000,864
Head params             : 7,738,036


# Optimizer and Mixed Precision Setup

In [17]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY
)

scaler = torch.amp.GradScaler("cuda") if (CFG.USE_AMP and CFG.DEVICE.type == "cuda") else None

# One-Epoch Training Function

In [18]:
def train_one_epoch(model, loader, optimizer, device, scaler=None, use_amp=False):
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        if use_amp and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda"):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            optimizer.step()

        total_loss += losses.item()

    return total_loss / max(len(loader), 1)

# Validation Loss Function

In [19]:
@torch.no_grad()
def validate_loss(model, loader, device):
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()

    return total_loss / max(len(loader), 1)

# Inference Latency Measurement

In [20]:
@torch.no_grad()
def measure_latency(model, loader, device, num_batches=20):
    model.eval()
    times = []

    for i, (images, _) in enumerate(loader):
        if i >= num_batches:
            break

        images = [img.to(device) for img in images]

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.time()
        _ = model(images)

        if device.type == "cuda":
            torch.cuda.synchronize()

        end = time.time()
        times.append((end - start) / len(images))

    return float(np.mean(times)) if len(times) > 0 else float("nan")

# Training Loop and Checkpoint Saving

In [21]:
best_val_loss = float("inf")

for epoch in range(CFG.EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        CFG.DEVICE,
        scaler=scaler,
        use_amp=CFG.USE_AMP
    )

    val_loss = validate_loss(model, val_loader, CFG.DEVICE)

    print(f"\nEpoch [{epoch+1}/{CFG.EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CFG.BEST_MODEL_PATH)
        print("Saved best model.")


Epoch [1/1]
Train Loss: 0.5154
Val Loss  : 0.3319
Saved best model.


# Load the Best Checkpoint

In [22]:
state_dict = torch.load(CFG.BEST_MODEL_PATH, map_location=CFG.DEVICE)
model.load_state_dict(state_dict)
model.to(CFG.DEVICE)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(384,), max_size=384, mode='bilinear')
  )
  (backbone): DetectorBackbone(
    (features): StudentNet_Pretrained(
      (features): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_siz

# Lightweight VOC-Style mAP@0.50 Evaluation

In [23]:
@torch.no_grad()
def evaluate_map_placeholder(model, loader, device, iou_thresh=0.5, score_thresh=0.05):
    model.eval()

    def compute_iou(box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        inter_w = max(0.0, x2 - x1)
        inter_h = max(0.0, y2 - y1)
        inter = inter_w * inter_h

        area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
        area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
        union = area1 + area2 - inter

        if union <= 0:
            return 0.0
        return inter / union

    def compute_ap(recalls, precisions):
        recalls = np.concatenate(([0.0], recalls, [1.0]))
        precisions = np.concatenate(([0.0], precisions, [0.0]))

        for i in range(len(precisions) - 1, 0, -1):
            precisions[i - 1] = max(precisions[i - 1], precisions[i])

        idx = np.where(recalls[1:] != recalls[:-1])[0]
        ap = np.sum((recalls[idx + 1] - recalls[idx]) * precisions[idx + 1])
        return ap

    all_gts = {}
    all_preds = []

    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for target, output in zip(targets, outputs):
            image_id = int(target["image_id"].item())

            gt_boxes = target["boxes"].cpu().numpy()
            gt_labels = target["labels"].cpu().numpy()

            all_gts[image_id] = {
                "boxes": gt_boxes,
                "labels": gt_labels
            }

            pred_boxes = output["boxes"].detach().cpu().numpy()
            pred_scores = output["scores"].detach().cpu().numpy()
            pred_labels = output["labels"].detach().cpu().numpy()

            for box, score, label in zip(pred_boxes, pred_scores, pred_labels):
                if score < score_thresh:
                    continue
                all_preds.append({
                    "image_id": image_id,
                    "box": box,
                    "score": float(score),
                    "label": int(label)
                })

    gt_classes = set()
    for v in all_gts.values():
        for lbl in v["labels"]:
            gt_classes.add(int(lbl))

    if len(gt_classes) == 0:
        return {"mAP50": 0.0, "mAP50_95": None}

    ap_per_class = {}

    for cls in sorted(gt_classes):
        cls_preds = [p for p in all_preds if p["label"] == cls]
        cls_preds.sort(key=lambda x: x["score"], reverse=True)

        npos = 0
        gt_used = {}

        for image_id, gt in all_gts.items():
            cls_gt_mask = (gt["labels"] == cls)
            cls_gt_boxes = gt["boxes"][cls_gt_mask]
            npos += len(cls_gt_boxes)
            gt_used[image_id] = np.zeros(len(cls_gt_boxes), dtype=bool)

        if npos == 0:
            continue

        tp = np.zeros(len(cls_preds))
        fp = np.zeros(len(cls_preds))

        for i, pred in enumerate(cls_preds):
            image_id = pred["image_id"]
            pred_box = pred["box"]

            gt = all_gts.get(image_id, None)
            if gt is None:
                fp[i] = 1
                continue

            cls_gt_mask = (gt["labels"] == cls)
            cls_gt_boxes = gt["boxes"][cls_gt_mask]

            if len(cls_gt_boxes) == 0:
                fp[i] = 1
                continue

            ious = np.array([compute_iou(pred_box, gt_box) for gt_box in cls_gt_boxes])
            best_idx = np.argmax(ious)
            best_iou = ious[best_idx]

            if best_iou >= iou_thresh:
                if not gt_used[image_id][best_idx]:
                    tp[i] = 1
                    gt_used[image_id][best_idx] = True
                else:
                    fp[i] = 1
            else:
                fp[i] = 1

        tp_cum = np.cumsum(tp)
        fp_cum = np.cumsum(fp)

        recalls = tp_cum / max(npos, 1e-8)
        precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-8)

        ap = compute_ap(recalls, precisions)
        ap_per_class[cls] = ap

    mAP50 = float(np.mean(list(ap_per_class.values()))) if len(ap_per_class) > 0 else 0.0

    return {
        "mAP50": mAP50,
        "mAP50_95": None,
        "AP_per_class": ap_per_class
    }

# Final Evaluation and Latency Report

In [24]:
metrics = evaluate_map_placeholder(model, val_loader, CFG.DEVICE)
latency = measure_latency(model, val_loader, CFG.DEVICE)

print("Validation Results")
print(f"mAP50   : {metrics['mAP50']:.4f}")
print(f"Latency : {latency * 1000:.3f} ms/image")
print(f"FPS     : {1.0 / latency:.2f}")

print("\nAP per class:")
for cls_id, ap in sorted(metrics["AP_per_class"].items()):
    cls_name = IDX_TO_CLASS.get(int(cls_id), f"class_{cls_id}")
    print(f"{int(cls_id):2d} ({cls_name:15s}): {float(ap):.4f}")

Validation Results
mAP50   : 0.1840
Latency : 7.494 ms/image
FPS     : 133.44

AP per class:
 1 (aeroplane      ): 0.3247
 2 (bicycle        ): 0.2960
 3 (bird           ): 0.0823
 4 (boat           ): 0.0000
 5 (bottle         ): 0.0000
 6 (bus            ): 0.2260
 7 (car            ): 0.1514
 8 (cat            ): 0.5637
 9 (chair          ): 0.0984
10 (cow            ): 0.0245
11 (diningtable    ): 0.1417
12 (dog            ): 0.3497
13 (horse          ): 0.2161
14 (motorbike      ): 0.1931
15 (person         ): 0.2968
16 (pottedplant    ): 0.0000
17 (sheep          ): 0.1821
18 (sofa           ): 0.1810
19 (train          ): 0.1617
20 (tvmonitor      ): 0.1898


# Save Final Weights

In [25]:
torch.save(model.state_dict(), CFG.FINAL_WEIGHTS_PATH)
print("Saved final weights to:", CFG.FINAL_WEIGHTS_PATH)

Saved final weights to: /kaggle/working/final_model_weights_only.pth


# File Submission

In [26]:
@torch.no_grad()
def export_submission_csv(model, loader, device, output_csv, score_thresh=0.05):
    model.eval()
    rows = []

    for images, image_ids in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for image_id, output in zip(image_ids, outputs):
            pred_boxes = output["boxes"].detach().cpu().numpy()
            pred_scores = output["scores"].detach().cpu().numpy()
            pred_labels = output["labels"].detach().cpu().numpy()

            pred_parts = []

            for box, score, label in zip(pred_boxes, pred_scores, pred_labels):
                if float(score) < score_thresh:
                    continue

                xmin, ymin, xmax, ymax = map(float, box.tolist())

                if xmax <= xmin or ymax <= ymin:
                    continue

                pred_parts.extend([
                    str(int(label)),
                    f"{float(score):.6f}",
                    f"{xmin:.6f}",
                    f"{ymin:.6f}",
                    f"{xmax:.6f}",
                    f"{ymax:.6f}",
                ])

            pred_str = " ".join(pred_parts)
            if pred_str == "":
                pred_str = "__EMPTY__"

            rows.append({
                "image_id": str(image_id),
                "PredictionString": pred_str,
                "Usage": "Public",
            })

    submission_df = pd.DataFrame(rows, columns=["image_id", "PredictionString", "Usage"])
    submission_df.insert(0, "id", range(len(submission_df)))
    submission_df.to_csv(output_csv, index=False)

    print("Saved submission file to:", output_csv)
    print("Total images in submission:", len(submission_df))
    print("Images with no predictions:", int((submission_df["PredictionString"] == "__EMPTY__").sum()))

    return submission_df

In [27]:
model.to(CFG.DEVICE)
model.eval()
submission_df = export_submission_csv(
    model=model,
    loader=test_loader,
    device=CFG.DEVICE,
    output_csv=CFG.SUBMISSION_PATH,
    score_thresh=0.05,
)

submission_df.head()
print(submission_df.isnull().sum())
print((submission_df["PredictionString"] == "__EMPTY__").sum())

Saved submission file to: /kaggle/working/submission.csv
Total images in submission: 5823
Images with no predictions: 242
id                  0
image_id            0
PredictionString    0
Usage               0
dtype: int64
242


# ONNX Conversion

In [28]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.1 MB/s eta 0:00:00


In [29]:
import torch

# ── 1. Prepare model ──────────────────────────────────────────────────────────
model.eval()
model.to("cpu")

# ── 2. Dummy input ────────────────────────────────────────────────────────────
dummy_input = [torch.randn(3, 384, 384)]

# ── 3. Export using legacy exporter (required for Faster R-CNN) ───────────────
torch.onnx.export(
    model,
    (dummy_input,),
    "model.onnx",
    opset_version=11,
    dynamo=False,              # ← forces legacy TorchScript path, avoids the NMS error
    do_constant_folding=True,
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
)

print("Exported to model.onnx")

/tmp/ipykernel_24/2146322373.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:4785: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
/usr/local/lib/python3.12/dist-packages/torchvision/ops/boxes.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torc

Exported to model.onnx
